# 148. Sort List

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** linked-list, two-pointers,
divide-and-conquer, merge-sort &nbsp;|&nbsp;
[LeetCode](https://leetcode.com/problems/sort-list/)

Given the `head` of a linked list, return *the list after sorting it in
**ascending order***.

---

### Example 1

```
Input:  head = [4,2,1,3]
Output: [1,2,3,4]
```

### Example 2

```
Input:  head = [-1,5,3,4,0]
Output: [-1,0,3,4,5]
```

### Example 3

```
Input:  head = []
Output: []
```

---

### Constraints

- The number of nodes in the list is in the range `[0, 5 * 10^4]`.
- `-10^5 <= Node.val <= 10^5`

**Follow-up:** can you sort the list in `O(n log n)` time **and** `O(1)` memory?

Two weeks of linked lists comes due here. #206 taught you to save `next` before
you overwrite it; this problem punishes forgetting in a new way - not a lost
tail, but a list with no end. #707 taught you that one node holding no data
deletes an `if` from four methods; here the same node deletes four `if`s from one
loop. And the divide-and-conquer shape is #105's, on a structure that will not
let you jump to the middle.

## Before you write anything

The sort is not the hard part - you already know merge sort. What is hard is that
every operation an array sort takes for granted (`arr[i]`, `len(arr)`,
`arr[lo:mid]`) costs `O(n)` here or does not exist at all. Work these out before
you write a line.

**1.** Name the sort you would reach for on a Python list, then price its inner
loop here. Quicksort partitions around a pivot; heapsort does index arithmetic;
binary insertion binary-searches. What does `arr[i]` cost on a linked list, and
how many of those three survive that? Merge sort does survive - say precisely
which property of *merging* is the reason.

**2.** The solution that passes and teaches nothing: walk the list into a Python
list, `sorted()` it, write the values back into the same nodes in order. Give its
time and space. It is accepted - LeetCode cannot tell. Say which of the two
follow-up requirements it fails, and why that one is the whole point of the
problem.

**3.** **Merging is the easy half - write it first.** You have two lists, each
already sorted, and you want one sorted list built by re-pointing arrows and
allocating no new nodes. Sketch the loop. Now count the `if`s: one to pick the
smaller front, one for "is `a` empty", one for "is `b` empty", and one for "is
this the first node of the result, or do I attach to the tail?". Three of those
four can be deleted. Which trick from #707 deletes the fourth, and what is the
*single* assignment that handles "one list is empty and the other still has 900
nodes"?

**4.** **Splitting is the half with the bug in it.** Slow moves one, fast moves
two, and when fast falls off the end slow is standing on the middle. Fine. Now:
you have the middle node, and you want two lists. What do you have to do to the
node **before** it? Skip that step and describe what the next recursive call
receives - and be specific about whether you get a wrong answer or a hang.

**5.** **`[2, 1]` decides your initialisation.** Trace slow/fast on a two-node
list, twice:

```
start with   slow = head, fast = head
start with   slow = head, fast = head.next
```

One splits it into `[2]` and `[1]`. The other splits it into `[]` and `[2, 1]`.
Work out which is which, then say what the losing one does on the recursive call
it makes. This one test case is most of the bug surface of this problem.

**6.** **Base case.** Which lists are already sorted with zero work? Write the
guard in one line. Then say why it is not an optimisation you could drop - what
does the recursion do without it?

**7.** **The follow-up is a lie unless you count the stack.** Route A allocates
no nodes, so it *looks* like `O(1)` memory. How deep does the recursion go for
`n = 50 000`? That is your real space bound. To get to genuine `O(1)` you stop
splitting top-down and merge bottom-up: merge every adjacent pair of sorted runs
of length 1, then of length 2, then 4, 8, ... What two things does that loop have
to track by hand that the call stack was tracking for you?

**8.** **Testing.** Unlike #707, `sortList` returns something, so you are not
blind. But three failures still *look* like success at a glance: the values come
out sorted with a node **duplicated**, the values come out sorted with a node
**dropped**, and the one that ruins your afternoon - the returned list **loops
back on itself**, so anything that reads it never stops. What do you check after
every call, and how do you read a list that might be circular without hanging?
(The harness below does all three. Guess them first.)

## Two routes - A to submit, B for the follow-up

**A - top-down merge sort** *(write this first)*
Three small pieces:

- `sortList(head)` - if the list has 0 or 1 nodes, return it unchanged.
  Otherwise: split, sort each half, merge the results.
- a split - slow/fast to the middle, **cut**, hand back two heads.
- a merge - a dummy node, a `tail` pointer, and one loop.

Costs: `O(n log n)` time (`log n` levels, `O(n)` of merging at each level),
`O(log n)` space for the call stack, no new nodes. That is accepted, and it is
the version to be able to write from memory.

The dummy in the merge is #707's sentinel wearing a different hat. Without it,
every iteration has to ask "am I the first node of the result?" and you carry a
`if head is None` branch through the entire loop. With it, the body is
`tail.next = smaller; tail = tail.next` with no branch at all, and you return
`dummy.next`. Same move, same payoff: **pay once, in state, to stop paying in
special cases.**

**B - bottom-up merge sort** *(the follow-up, genuinely `O(1)`)*
No recursion, so no stack. Count `n` once. Then for `width = 1, 2, 4, 8, ...`
while `width < n`: walk the list once, and at each step chop off two runs of
`width` nodes, merge them, and hang the result off the tail of what you have
built so far. One dummy for the whole pass.

You reuse the *same* merge from route A - that is the point of writing it as its
own function. What is new is a `cut(head, k)` helper: detach the first `k` nodes
and return the head of the rest. Every bug in route B is either an off-by-one in
a run length or a `next` you forgot to set to `None` - the same two families of
bug as #707, in a new costume.

> **The cut is the whole problem.** Not the merging, not the middle-finding.
> Splitting an array is `arr[:mid]` and costs you nothing but a copy. Splitting a
> linked list means **severing** it, because a list has no length field to say
> where it ends - it has exactly one end marker, `None`, and if you do not write
> one at the end of the first half then both halves are the same list. The
> recursion then never shrinks, and Python reports it as `RecursionError` a few
> thousand frames from where you made the mistake.

Write A, run the tests, then write B and run the same tests.

In [4]:
# Definition for singly-linked list.
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class Stack:
    def __init__(self, head=None, tail=None):
        self.head = head
        self.tail = tail

    def addInHead(self, val):
        newNode = ListNode(val)

        if self.head is None:
            self.head = newNode
            self.tail = newNode
            return

        newNode.next = self.head
        self.head = newNode

    def addInTail(self, val):
        newNode = ListNode(val)

        if self.head is None:
            self.head = newNode
            self.tail = newNode
            return

        self.tail.next = newNode
        self.tail = newNode

    def add(self,val):
        if self.head is None or self.head.val >= val:
            self.addInHead(val)
            return
        if self.tail.val <= val:
            self.addInTail(val)
            return
        i = ListNode(val)
        prev = self.head
        current = self.head.next
        while current and current.val <= val:
            prev = current
            current = current.next
        prev.next = i
        i.next = current
        return



class Solution:
    def sortList(self, head: ListNode) -> ListNode:
        if head is None : return None
        stack:Stack = Stack()
        current = head
        while current :
            stack.add(current.val)
            current = current.next
        return stack.head


In [9]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class Stack:
    def __init__(self, head=None, tail=None):
        self.head = head
        self.tail = tail

    def addInHead(self, val):
        if self.head is None:
            self.head = val
            self.tail = val
            val.next = None
            return

        val.next = self.head
        self.head = val

    def addInTail(self, Val):
        Val.next = None
        if self.head is None:
            self.head = Val
            self.tail = Val

            return

        self.tail.next = Val
        self.tail = Val


    def add(self,Val):
        if self.head is None or self.head.val >= Val.val:
            self.addInHead(Val)
            return
        if self.tail.val <= Val.val:
            self.addInTail(Val)
            return
        prev = self.head
        current = self.head.next
        while current and current.val <= Val.val:
            prev = current
            current = current.next
        prev.next = Val
        Val.next = current
        return



class Solution:
    def sortList(self, head: ListNode) -> ListNode:
        if head is None : return None
        stack:Stack = Stack()
        current = head
        while current :
            Next = current.next
            stack.add(current)
            current = Next
        return stack.head

### The test harness

Question 8's answer. `sortList` returns a head, so a wrong *order* is obvious -
but the three interesting failures are not about order.

So `check` does three things after every call. It reads the returned chain with a
walk that remembers every node it has already passed, so a list that loops back
on itself is **reported**, not hung on - that is the failure that would otherwise
freeze the notebook. It compares the values against `sorted(values)`, which
catches a dropped or duplicated node as a length mismatch. And it compares the
set of node objects that come out against the set that went in, so it can tell
you whether you re-spliced the original nodes or quietly allocated new ones -
not a failure, but the difference between `O(1)` and `O(n)` extra space, and the
harness is the only place you will ever see it.

Exceptions are caught and reported as a FAIL line, so a `RecursionError` from a
forgotten cut shows up as a normal test result. The big cases are timed: an
`O(n^2)` sort gets the right answer on all 50 000 nodes eventually, and the clock
is the only thing that tells you it was the wrong algorithm.

Run this cell; don't edit it.

In [7]:
import random
import time
from itertools import permutations


def build(values):
    """Python list -> linked list, return the head."""
    head = None
    for v in reversed(values):        # back-to-front, so order is preserved
        head = ListNode(v, head)
    return head


def walk(head, limit=200_000):
    """Collect the node objects in the chain.

    Remembers what it has already seen, so a circular list returns
    (nodes_so_far, True) instead of looping forever.
    """
    out, seen, cur = [], set(), head
    while cur is not None:
        if id(cur) in seen:
            return out, True
        if len(out) >= limit:         # longer than any legal test list
            return out, True
        seen.add(id(cur))
        out.append(cur)
        cur = cur.next
    return out, False


def check(values):
    """Sort a copy of `values` as a linked list and verify the result.

    Returns (ok, message).
    """
    head = build(values)
    before, _ = walk(head)
    came_in = {id(n) for n in before}

    try:
        t0 = time.perf_counter()
        out = Solution().sortList(head)
        ms = (time.perf_counter() - t0) * 1000
    except RecursionError:
        return False, "RecursionError - the recursion never reached the base case (did you cut?)"
    except Exception as e:
        return False, f"raised {type(e).__name__}: {e}"

    nodes, looped = walk(out)
    got = [n.val for n in nodes]
    want = sorted(values)

    if looped:
        return False, (f"the returned list never ends - read {len(got)} nodes and came back "
                       f"to one already visited")

    if got != want:
        if len(got) != len(want):
            kind = "dropped" if len(got) < len(want) else "duplicated"
            return False, (f"got {len(got)} nodes, expected {len(want)} - {kind} nodes\n"
                           f"       got  {got[:12]}{' ...' if len(got) > 12 else ''}\n"
                           f"       want {want[:12]}{' ...' if len(want) > 12 else ''}")
        i = next(i for i, (g, w) in enumerate(zip(got, want)) if g != w)
        lo, hi = max(0, i - 3), i + 4
        return False, (f"first wrong value at index {i}\n"
                       f"       got  {got[lo:hi]}\n"
                       f"       want {want[lo:hi]}")

    reused = came_in >= {id(n) for n in nodes} if nodes else True
    how = "re-spliced the original nodes" if reused else "allocated new nodes"
    return True, f"{len(got):6d} nodes {ms:8.1f} ms   {how}"


def report(name, ok, msg):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if msg:
        print(f"       {msg}")

In [8]:
# tests
CASES = [
    ("the LeetCode example 1",              [4, 2, 1, 3]),
    ("the LeetCode example 2",              [-1, 5, 3, 4, 0]),
    ("empty list",                          []),
    ("one node",                            [7]),
    ("question 5: two nodes, sorted",       [1, 2]),
    ("question 5: two nodes, NOT sorted",   [2, 1]),
    ("two nodes, equal",                    [3, 3]),
    ("already sorted, 20 nodes",            list(range(20))),
    ("exactly reversed, 20 nodes",          list(range(20))[::-1]),
    ("every value identical",               [5] * 10),
    ("duplicates everywhere",               [3, 1, 3, 2, 1, 2, 3, 1]),
    ("negatives, zero, and the ceilings",   [-1, 0, -100000, 100000, 0, -100000]),
    ("length 16 - a power of two",          [(i * 7) % 16 for i in range(16)]),
    ("length 17 - deliberately not one",    [(i * 7) % 17 for i in range(17)]),
    ("length 1023 - odd splits all the way down", [(i * 31) % 997 for i in range(1023)]),
]

for name, values in CASES:
    report(name, *check(values))

# all six orderings of three elements - the smallest case with a real split
bad = [p for p in permutations([1, 2, 3]) if not check(list(p))[0]]
report("all 6 permutations of [1,2,3]", not bad, f"failed on {bad}" if bad else "all six sorted")

# random sequences, including heavy duplicates and long runs
for n, seed, hi in [(50, 1, 1000), (200, 2, 10), (1000, 3, 100000), (5000, 4, 50)]:
    random.seed(seed)
    values = [random.randint(-hi, hi) for _ in range(n)]
    report(f"random: {n} values in [-{hi}, {hi}] (seed {seed})", *check(values))

# the constraint ceiling: 5 * 10^4 nodes. An O(n^2) sort finishes - watch the clock.
random.seed(148)
report("ceiling: 50 000 random values", *check([random.randint(-100000, 100000) for _ in range(50000)]))
report("ceiling: 50 000 already sorted", *check(list(range(50000))))
report("ceiling: 50 000 exactly reversed", *check(list(range(50000))[::-1]))
report("ceiling: 50 000 identical values", *check([0] * 50000))

# The shape LeetCode uses to kill insertion sort: ascending, with the max moved to the
# front. That one displaced element defeats a head shortcut AND a tail shortcut for the
# whole rest of the run, so an O(n^2) sort does the most work it possibly can here -
# about 1.25e9 steps. Note this list is one move away from "already sorted", which is the
# fastest case for the same code. That gap is the point.
report("ceiling: 50 000 sorted, max moved to the front",
       *check([50000] + list(range(1, 50000))))

OK   the LeetCode example 1
            4 nodes      0.0 ms   re-spliced the original nodes
OK   the LeetCode example 2
            5 nodes      0.0 ms   re-spliced the original nodes
OK   empty list
            0 nodes      0.0 ms   re-spliced the original nodes
OK   one node
            1 nodes      0.0 ms   re-spliced the original nodes
OK   question 5: two nodes, sorted
            2 nodes      0.0 ms   re-spliced the original nodes
OK   question 5: two nodes, NOT sorted
            2 nodes      0.0 ms   re-spliced the original nodes
OK   two nodes, equal
            2 nodes      0.0 ms   re-spliced the original nodes
OK   already sorted, 20 nodes
           20 nodes      0.0 ms   re-spliced the original nodes
OK   exactly reversed, 20 nodes
           20 nodes      0.0 ms   re-spliced the original nodes
OK   every value identical
           10 nodes      0.0 ms   re-spliced the original nodes
OK   duplicates everywhere
            8 nodes      0.0 ms   re-spliced the original node

## After it passes

- **Count your `if`s.** Write the merge once without the dummy node and once with
  it, and count the branches in the loop body of each. That number is the answer
  you give when someone asks why you allocate a node that never holds data - and
  it is the same answer you gave in #707.
- **Write route B and run the same tests.** Then time both at `n = 50 000` and
  compare. B removes `O(log n)` stack frames; does the wall clock actually drop,
  or did you trade recursion for bookkeeping at roughly par? Say why before you
  measure, then check.
- **Race the cheat.** Time the "dump to a Python list, `sorted()`, write the
  values back" version against your route A at `n = 50 000`. It will win, and not
  by a little - Timsort is C and your merge is a Python loop over objects. Write
  the two-sentence version of why the cheat is still the wrong answer to *this*
  problem. The words you want are in question 2.
- **Is your merge stable?** Find the single comparison that decides it - `<=`
  against `<` - and say which way round makes it stable. Then: for this problem,
  sorting bare integers, does stability change any output at all? Now suppose
  each node carried `(name, score)` and you sorted by score. Same question.
- **Print the recursion depth.** Add a `depth` parameter to route A, track the
  maximum, and run it on 50 000 nodes. Check it against the `log2(50000)` you
  predicted in question 7 - and against Python's default recursion limit of 1000,
  which is the number that tells you how much headroom this design actually had.
- **The invariant list**, same as #707. Write down what your sort promises:
  the output is sorted; every node that went in comes out exactly once; no node's
  `next` points backwards into the part already emitted; the last node's `next` is
  `None`. Then, for each of your three functions, name the ones it could break.
  That list is exactly what `check` above asserts, which is why a wrong answer is
  reported as *what* is wrong rather than as a hang.
- Siblings - and the first two are literally the two halves of this problem:
  **#21 Merge Two Sorted Lists** (your merge, alone, as its own problem),
  **#876 Middle of the Linked List** (your split, alone, minus the cut),
  #143 Reorder List (split + reverse + merge - all three of your linked-list
  notebooks in one function), #23 Merge k Sorted Lists (this merge, k at a time,
  and the reason heaps exist), #147 Insertion Sort List (the `O(n^2)` answer, and
  a fair question about when that is the right call).